# Milestone 2 - Document parsing & parent-child chunking

Builds the synthetic, de-identified enterprise corpus (PDF, DOCX, CSV, Markdown) together with the labelled QA sets, then parses every document **layout-aware**: tables are detected and converted to Markdown, headings define the section hierarchy, and documents are split into *parent* units (page / section / row - the unit we cite) and *child* chunks (paragraph groups or whole tables - the unit we embed). PII (e-mail addresses and phone numbers) is redacted before anything is indexed.

In [1]:
import os, sys, json, time, warnings, pathlib
warnings.filterwarnings("ignore")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
sys.path.insert(0, str(ROOT)); os.chdir(ROOT)
import pandas as pd
import matplotlib.pyplot as plt
pd.set_option("display.max_colwidth", 120)
from rag import config

In [2]:
from rag import corpus, parsing
summary = corpus.build_corpus()          # deterministic (seeded) - rebuilds identical files
summary

{'documents': ['CU4_Controller_Firmware_Release_Notes.docx',
  'CU4_Known_Issues.md',
  'HIS_Annual_Operations_Report_FY2025.pdf',
  'HIS_Employee_Handbook_2026.pdf',
  'HIS_Quality_Compliance_Manual.pdf',
  'HIS_Site_Energy_and_Sustainability_Review_2025.pdf',
  'HX_Pump_Selection_Guide.md',
  'HX_Pump_Series_Technical_Specification.docx',
  'Supplier_Audit_Report_2025.pdf',
  'engineering_meeting_notes.md',
  'equipment_maintenance_schedule.csv',
  'incident_postmortems.md',
  'maintenance_work_orders_2026.csv',
  'supplier_register.csv'],
 'facts': 1064,
 'test_pairs': 200,
 'train_pairs': 2063,
 'train_facts': 864,
 'unanswerable': 20}

In [3]:
docs = pd.DataFrame([{"document": p.name, "format": p.suffix[1:], "kb": round(p.stat().st_size / 1024, 1)}
                     for p in sorted(config.CORPUS_DIR.iterdir())])
docs

,document,format,kb
0,CU4_Controller_Firmware_Release_Notes.docx,docx,36.5
1,CU4_Known_Issues.md,md,1.6
2,HIS_Annual_Operations_Report_FY2025.pdf,pdf,12.2
3,HIS_Employee_Handbook_2026.pdf,pdf,9.7
4,HIS_Quality_Compliance_Manual.pdf,pdf,6.4
5,HIS_Site_Energy_and_Sustainability_Review_2025.pdf,pdf,9.6
6,HX_Pump_Selection_Guide.md,md,2.4
7,HX_Pump_Series_Technical_Specification.docx,docx,37.0
8,Supplier_Audit_Report_2025.pdf,pdf,7.5
9,engineering_meeting_notes.md,md,2.3


In [4]:
chunks = parsing.parse_corpus()
parsing.save_chunks(chunks)
df = pd.DataFrame([{**c.__dict__, "words": len(c.text.split())} for c in chunks])
print(f"{len(chunks)} child chunks from {df.parent_id.nunique()} parent units")
df.groupby(["doc_type", "kind"]).agg(chunks=("chunk_id", "count"), mean_words=("words", "mean")).round(1)

Consider using the pymupdf_layout package for a greatly improved page layout analysis.


273 child chunks from 241 parent units


chunks  mean_words
doc_type kind                     
csv      text      140        23.9
docx     table      12        75.0
         text       24        47.0
md       text       42        43.6
pdf      table      12        83.4
         text       43        60.0

### Table structure survives parsing
A table from a PDF page (detected geometrically by PyMuPDF) and one from a Word document:

In [5]:
print(next(c.text for c in chunks if c.doc_type == "pdf" and c.kind == "table"))
print()
print(next(c.text for c in chunks if c.doc_type == "docx" and c.kind == "table"))

[Annual Operations Report FY2025 | 2. Site Review: Tampere Works (Finland)]
| Metric | FY2025 |
|---|---|
| Units produced | 34,520 |
| On-time delivery | 98.3% |
| Lost-time incidents | 6 |
| Energy consumption | 30,660 MWh |
| Headcount | 189 |

[HX Centrifugal Pump Series - Technical Specification | 2. HX-110 Centrifugal Pump]
| Parameter | Value |
|---|---|
| Maximum flow rate | 1865 L/min |
| Maximum head | 74 m |
| Motor power | 14.1 kW |
| Impeller diameter | 272 mm |
| Dry weight | 1110 kg |
| Inlet connection | DN150 |
| Maximum fluid temperature | 120 degC |
| Motor enclosure rating | IP65 |


### Parent-child structure
One PDF page = one parent; its children are the paragraph groups and the table on that page.

In [6]:
pid = next(c.parent_id for c in chunks if c.doc.startswith("HIS_Annual") and c.page == 3)
for c in chunks:
    if c.parent_id == pid:
        print(f"--- child {c.chunk_id} ({c.kind})\n{c.text}\n")

--- child HIS_Annual_Operations_Report_FY2025.pdf::3::0 (text)
[Annual Operations Report FY2025 | 3. Site Review: Gdansk Assembly (Poland)]
## 3. Site Review: Gdansk Assembly (Poland) The Gdansk Assembly site in Poland is led by site director Mikael Ferreira. The table below summarises its key operating indicators for the fiscal year.

--- child HIS_Annual_Operations_Report_FY2025.pdf::3::1 (table)
[Annual Operations Report FY2025 | 3. Site Review: Gdansk Assembly (Poland)]
| Metric | FY2025 |
|---|---|
| Units produced | 15,710 |
| On-time delivery | 90.8% |
| Lost-time incidents | 0 |
| Energy consumption | 24,780 MWh |
| Headcount | 359 |

--- child HIS_Annual_Operations_Report_FY2025.pdf::3::2 (text)
[Annual Operations Report FY2025 | 3. Site Review: Gdansk Assembly (Poland)]
Improvement priorities for FY2026 include lean flow, energy efficiency and supplier quality.



### CSV rows and Markdown sections

In [7]:
print(next(c.text for c in chunks if c.doc_type == "csv"))
print()
print(next(c.text for c in chunks if c.doc_type == "md" and c.page == 1))

[Equipment Maintenance Schedule | Equipment Maintenance Schedule row 1]
asset_id: AST-2526; description: Paint booth; site: Graz Electronics; service_interval_days: 45; last_service_date: 2026-01-15; assigned_technician: Mikael Etxeberria

[CU-4 Controller Known Issues | Issue CU4-KI-101 (affects firmware 4.1)]
## Issue CU4-KI-101 (affects firmware 4.1) Symptom: Modbus registers above 40000 return stale values. Workaround: restart the controller during the next maintenance window. Resolved in firmware 4.2.


### PII redaction

In [8]:
raw = (config.CORPUS_DIR / "engineering_meeting_notes.md").read_text().split("## ")[1].splitlines()[2]
print("raw     :", raw)
print("indexed :", parsing.redact_pii(raw))
assert not any("@" in c.text for c in chunks), "e-mail address leaked into index"

raw     : Chair: Bruno Virtanen (bruno.virtanen@halvorsen-is.example, +358 40 931 7554). Attendees: 14.
indexed : Chair: Bruno Virtanen ([EMAIL], [PHONE]). Attendees: 14.


### Ground-truth validation
Every one of the labelled facts must literally appear on the page it is labelled with.

In [9]:
facts = corpus.load_jsonl(config.EVAL_DIR / "facts.jsonl")
v = parsing.validate_ground_truth(chunks, facts)
print(f"checked {v['checked']} facts - missing on labelled page: {v['missing']}")
test = pd.DataFrame(corpus.load_jsonl(config.EVAL_DIR / "test_qa.jsonl"))
print(f"held-out test set: {len(test)} QA pairs;  reranker training questions: {len(corpus.load_jsonl(config.EVAL_DIR / 'train_qa.jsonl'))}")
test.groupby("doc").size().rename("test pairs").to_frame()

checked 1064 facts - missing on labelled page: 0
held-out test set: 200 QA pairs;  reranker training questions: 2063


,test pairs
doc,
CU4_Controller_Firmware_Release_Notes.docx,16
CU4_Known_Issues.md,4
HIS_Annual_Operations_Report_FY2025.pdf,20
HIS_Employee_Handbook_2026.pdf,18
HIS_Quality_Compliance_Manual.pdf,12
HIS_Site_Energy_and_Sustainability_Review_2025.pdf,10
HX_Pump_Selection_Guide.md,10
HX_Pump_Series_Technical_Specification.docx,28
Supplier_Audit_Report_2025.pdf,12


In [10]:
test.sample(8, random_state=1)[["question", "answer", "doc", "page"]]

,question,answer,doc,page
58,What is the impeller diameter of the HX-200 pump?,416 mm,HX_Pump_Series_Technical_Specification.docx,5
40,How often is the QMS management review held?,every 6 months,HIS_Quality_Compliance_Manual.pdf,1
34,What was the on-time delivery rate at Graz Electronics?,87.3%,HIS_Annual_Operations_Report_FY2025.pdf,8
102,How long does Alpine Castings take to deliver after we place an order?,38,supplier_register.csv,12
184,At what flow does the HX-320 typically operate?,960 L/min,HX_Pump_Selection_Guide.md,7
198,Which release resolved CU4-KI-102?,4.3,CU4_Known_Issues.md,2
95,How does Polaris Polymers rate on quality in the supplier register?,94,supplier_register.csv,3
4,What is the laptop refresh cycle?,36 months,HIS_Employee_Handbook_2026.pdf,2
